# MLPC 2026 — Challenge Baseline: Sound Event Detection

Welcome to the MLPC 2026 challenge! This notebook implements a **baseline Sound Event Detection (SED) system** that serves as your starting point for the challenge.

In the previous project phases, you recorded and annotated audio data and implemented classifiers to predict whether a given sound event class is active within isolated 1-second audio segments. The goal of the current challenge is to **extend these classifiers to full sound event detection**: given an audio recording of arbitrary length, your system should predict not only *which* sound events occur, but also *when* — providing **onset and offset timestamps** for each predicted event.

### What this notebook covers

1. Loading and exploring the challenge dataset
2. Splitting the validation set into a local validation set and a non-hidden test set
3. Loading precomputed audio features and extracting segment-level labels
4. Training a multi-label decision-tree classifier (one tree per sound event class)
5. Applying the classifier to full recordings to produce SED predictions
6. Evaluating the system using the challenge metric (segment-level macro F1)
7. Generating predictions for the hidden test set (your final submission)

### Your task

The decision-tree baseline is intentionally simple — it is the **floor your system should beat**. Replace or extend it with better classifiers, better features, or more advanced temporal modeling.

## Outline

1. [Setup: Imports, Constants, and Paths](#1)
2. [Dataset Description](#2)
3. [Data Splits](#3)
4. [Feature Loading and Training Data Preparation](#4)
5. [Training the Classifier](#5)
6. [Sound Event Detection Inference](#6)
7. [Evaluation](#7)
8. [Generating Predictions for the Hidden Test Set](#8)

## 1. Setup

In [ ]:
import os
import sys
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.multioutput import MultiOutputClassifier
from typing import List, Tuple, Dict

# evaluate.py is located in the same directory as this notebook.
# Make sure to run the notebook from the challenge_baseline/ directory.
sys.path.insert(0, os.getcwd())
from evaluate import (
    aggregate_ground_truth_annotations,
    build_segment_frame_from_intervals,
    calculate_f1_score,
)

In [ ]:
# The precomputed audio features stored in each .npz file.
# These are the same features as in the classification tutorial.
FEATURE_NAMES = [
    "zcr_mean",        "zcr_std",        "zcr_min",        "zcr_max",
    "melspect_mean",   "melspect_std",   "melspect_min",   "melspect_max",
    "mfcc_mean",       "mfcc_std",       "mfcc_min",       "mfcc_max",
    "mfcc_d_mean",     "mfcc_d_std",     "mfcc_d_min",     "mfcc_d_max",
    "mfcc_d2_mean",    "mfcc_d2_std",    "mfcc_d2_min",    "mfcc_d2_max",
    "flux_mean",       "flux_std",       "flux_min",       "flux_max",
    "flatness_mean",   "flatness_std",   "flatness_min",   "flatness_max",
    "centroid_mean",   "centroid_std",   "centroid_min",   "centroid_max",
    "bandwidth_mean",  "bandwidth_std",  "bandwidth_min",  "bandwidth_max",
    "contrast_mean",   "contrast_std",   "contrast_min",   "contrast_max",
    "rolloff_low_mean",  "rolloff_low_std",  "rolloff_low_min",  "rolloff_low_max",
    "rolloff_high_mean", "rolloff_high_std", "rolloff_high_min", "rolloff_high_max",
    "energy_mean",     "energy_std",     "energy_min",     "energy_max",
    "power_mean",      "power_std",      "power_min",      "power_max",
]

# The 15 target sound event classes — sorted alphabetically to match the .npz annotation order.
CLASS_NAMES = [
    "bell_ringing",
    "coffee_machine",
    "cutlery_dishes",
    "door_open_close",
    "footsteps",
    "keyboard_typing",
    "keychain",
    "light_switch",
    "microwave",
    "phone_ringing",
    "running_water",
    "toilet_flushing",
    "vacuum_cleaner",
    "wardrobe_drawer_open_close",
    "window_open_close",
]

SEGMENT_LENGTH = 1.0  # each feature vector covers a 1-second window
HOP_SIZE       = 0.5  # segments are extracted with 50% overlap

In [ ]:
# TODO: Change this path to wherever you extracted the challenge dataset.
PATH_TO_DATASET = "./"

PATH_TRAIN = os.path.join(PATH_TO_DATASET, "train")
PATH_VAL   = os.path.join(PATH_TO_DATASET, "validation")
PATH_TEST  = os.path.join(PATH_TO_DATASET, "test")

# Sanity check — will raise an AssertionError if a path does not exist.
for path in [PATH_TRAIN, PATH_VAL, PATH_TEST]:
    assert os.path.isdir(path), f"Directory not found: {path}"
for path in [
    os.path.join(PATH_TRAIN, "annotations.csv"),
    os.path.join(PATH_TRAIN, "audio_features"),
    os.path.join(PATH_VAL,   "annotations.csv"),
    os.path.join(PATH_VAL,   "audio_features"),
    os.path.join(PATH_TEST,  "audio_features"),
]:
    assert os.path.exists(path), f"Path not found: {path}"

print("Dataset paths OK.")

<a id='2'></a>
## 2. Dataset Description

The challenge dataset is organized into three splits:

```
MLPC2026_challenge/
├── train/
│   ├── metadata.csv          ← recording metadata (device, environment, ...)
│   ├── annotations.csv       ← ground-truth event annotations (onset, offset, annotator)
│   └── audio_features/
│       ├── 000001.npz        ← precomputed features for one recording
│       ├── 000002.npz
│       └── ...
├── validation/
│   ├── metadata.csv
│   ├── annotations.csv
│   └── audio_features/
│       └── ...
└── test/
    └── audio_features/       ← NO annotations — this is the hidden evaluation set
        └── ...
```

### Feature files (`.npz`)

Each `.npz` file corresponds to one audio recording and contains precomputed features for **all overlapping 1-second segments**, extracted with a hop size of 0.5 s. That is, a 30-second recording produces segments starting at t = 0.0, 0.5, 1.0, 1.5, ..., 29.5 seconds — a total of 60 segments.

| Key | Shape | Description |
|-----|-------|-------------|
| `start_time` | `(N,)` | Start time of each segment (0.0, 0.5, 1.0, ...) |
| `end_time` | `(N,)` | End time of each segment (1.0, 1.5, 2.0, ...) |
| `annotations` | `(N, 15, 2)` | Per-segment annotation scores for each class and annotator |
| `mfcc_mean` | `(N, 32)` | Mean MFCC coefficients per segment (similarly for all other features) |
| ... | ... | ... |

The `annotations` field is only present in the **train** and **validation** splits. The **test** split contains only features.

### Annotation encoding

For each segment `t` and class `c`, `annotations[t, c, :]` holds two values — one per annotator — each in `[0, 1]`. The value represents the **fraction of the 1-second segment** during which that annotator marked the class as active. A value of `0.0` means completely silent; `1.0` means active throughout the entire segment.

To obtain binary training labels, we check whether each fraction is greater than zero (any overlap with the annotation) and then aggregate across annotators via majority vote.

In [ ]:
train_files = sorted(glob.glob(os.path.join(PATH_TRAIN, "audio_features", "*.npz")))
val_files   = sorted(glob.glob(os.path.join(PATH_VAL,   "audio_features", "*.npz")))
test_files  = sorted(glob.glob(os.path.join(PATH_TEST,  "audio_features", "*.npz")))

print(f"Training recordings:   {len(train_files)}")
print(f"Validation recordings: {len(val_files)}")
print(f"Test recordings:       {len(test_files)}  (hidden — no ground-truth labels)")

<a id='3'></a>
## 3. Data Splits

The challenge dataset comes with three pre-defined splits:

| Split | Size  | Annotations? | Purpose |
|-------|-------|-------------|--------|
| `train` | 3 704 | ✓ | Training your models |
| `validation` | 999   | ✓ | Tuning and local evaluation |
| `test` | 1 007 | ✗ | **Hidden evaluation set** — predictions submitted for final ranking |

As we can see in the following output cells, samples of the training and validation set contain more information (like annotations!) compared to the test set.

In [ ]:
training_sample = dict(np.load(train_files[0], allow_pickle=True))
training_sample.keys()

In [ ]:
validation_sample = dict(np.load(val_files[0], allow_pickle=True))
validation_sample.keys()

In [ ]:
test_sample = dict(np.load(test_files[0], allow_pickle=True))
test_sample.keys()

### Why we need an additional non-hidden test set

The `test` split has no ground-truth labels: you can generate predictions for it, but you cannot evaluate them locally. This is intentional — it prevents tuning your model on the evaluation data.

However, you also need a held-out set that you can evaluate locally without bias. If you tune your model based on validation-set performance, those metrics become overly optimistic. A **separate local test set** gives you an unbiased estimate of your model's true performance before final submission.

We therefore split the 999 validation recordings into two halves:
- **Local validation set** (500 recordings): use for hyperparameter tuning and model comparison.
- **Non-hidden test set** (499 recordings): use as a final sanity check before submitting. Once you evaluate on this set, do not use the result to tune your model — otherwise it is no longer unbiased.

The hidden `test` split is used only to generate your final submission CSV.

In [ ]:
rng = np.random.default_rng(seed=42)

val_files_shuffled = rng.permutation(val_files).tolist()
n_val = len(val_files_shuffled) // 2

our_val_files  = val_files_shuffled[:n_val]   # local validation set
our_test_files = val_files_shuffled[n_val:]   # non-hidden local test set

print(f"Local validation set:  {len(our_val_files)} recordings")
print(f"Non-hidden test set:   {len(our_test_files)} recordings")
print(f"Hidden test set:       {len(test_files)} recordings")

<a id='4'></a>
## 4. Feature Loading and Training Data Preparation

Before training, we need two things for each recording in the training set:
1. A **feature matrix** `X` of shape `(N_segments, D_features)` — one row per segment, one column per feature dimension.
2. A **label matrix** `Y` of shape `(N_segments, 15)` — one binary value per segment and class.

For training, we treat all segments from all recordings as **independent examples** and simply stack them. The temporal order within a recording does not matter at training time.

In [ ]:
def build_feature_matrix(data: dict) -> np.ndarray:
    """Concatenate all named features from a loaded .npz dict into one matrix.

    Parameters
    ----------
    data : dict
        Contents of a .npz file (loaded with allow_pickle=True).

    Returns
    -------
    X : np.ndarray of shape (N_segments, D_features)
    """
    arrays = []
    for feat_name in FEATURE_NAMES:
        feat = data[feat_name]
        if feat.ndim == 1:
            feat = feat[:, np.newaxis]
        arrays.append(feat.astype(np.float32))
    return np.concatenate(arrays, axis=1)


def get_segment_labels(data: dict) -> np.ndarray:
    """Extract binary multilabel targets from the annotations field of a .npz file.

    The annotations array has shape (N_segments, N_classes, N_annotators).
    Each value is the fraction of the 1-second segment during which a given
    annotator marked the class as active.

    We binarize by checking for any overlap (> 0) and then apply majority vote
    across annotators to obtain the final binary label.

    Parameters
    ----------
    data : dict
        Contents of a .npz file (loaded with allow_pickle=True).
        Must contain the 'annotations' key.

    Returns
    -------
    Y : np.ndarray of shape (N_segments, N_classes), dtype int
    """
    annotations = data["annotations"]           # shape: (N, C, A)
    # Binarize: 1 if annotator marked ANY part of this segment as active.
    # (This threshold can be improved, e.g. > 0.5 for majority of the segment.)
    binary = (annotations > 0).astype(int)      # (N, C, A)
    # Majority vote: class is active if more than half the annotators agree.
    votes = binary.sum(axis=2)                  # (N, C)
    n_annotators = binary.shape[2]
    y = (votes > (n_annotators // 2)).astype(int)
    return y


def load_all_segments(
    file_list: List[str],
) -> Tuple[np.ndarray, np.ndarray]:
    """Load features and labels from a list of .npz files and stack them.

    Parameters
    ----------
    file_list : list of str
        Paths to .npz audio feature files.

    Returns
    -------
    X : np.ndarray of shape (N_total_segments, D_features)
    Y : np.ndarray of shape (N_total_segments, N_classes)
    """
    X_list, Y_list = [], []
    for filepath in file_list:
        data = dict(np.load(filepath, allow_pickle=True))
        X_list.append(build_feature_matrix(data))
        Y_list.append(get_segment_labels(data))
    return np.vstack(X_list), np.vstack(Y_list)

In [ ]:
# Inspect one training file to verify the loading functions
example_data = dict(np.load(train_files[0], allow_pickle=True))
X_ex = build_feature_matrix(example_data)
Y_ex = get_segment_labels(example_data)

print(f"File:              {os.path.basename(train_files[0])}")
print(f"Number of segments:  {X_ex.shape[0]}")
print(f"Feature dimension:   {X_ex.shape[1]}")
print(f"Number of classes:   {Y_ex.shape[1]}")
print(f"Segment start times (first 6): {example_data['start_time'][:6]} ...  (hop = 0.5 s)")
print()
print("Active classes per segment (first 10 segments):")
for i in range(min(10, len(Y_ex))):
    active = [CLASS_NAMES[c] for c in range(len(CLASS_NAMES)) if Y_ex[i, c] == 1]
    t = example_data['start_time'][i]
    print(f"  t = {t:.1f} s : {active if active else '(none)'}")

In [ ]:
# Load training segments from all (or a subset of) training recordings.
#
# MAX_TRAINING_FILES    — None = use all ~3 700 training files.
#                         Set to e.g. 500 for a quick test run.
# MAX_TRAINING_SEGMENTS — After loading, randomly subsample to this many segments.
#                         Keeps training fast even when loading all files.
#                         None = keep all loaded segments (can be slow to train).
#
# Tip: A decision tree does not need hundreds of thousands of examples to converge.
# 50 000 segments drawn from diverse recordings is a good default starting point.
MAX_TRAINING_FILES    = None    # None = use all training files
MAX_TRAINING_SEGMENTS = 50_000  # None = keep all loaded segments

train_files_to_use = train_files
if MAX_TRAINING_FILES is not None:
    train_files_to_use = rng.choice(
        train_files, size=min(MAX_TRAINING_FILES, len(train_files)), replace=False
    ).tolist()

print(f"Loading features from {len(train_files_to_use)} training recordings...")
X_train, Y_train = load_all_segments(train_files_to_use)
print(f"Loaded {X_train.shape[0]} segments with {X_train.shape[1]} features each.")

if MAX_TRAINING_SEGMENTS is not None and X_train.shape[0] > MAX_TRAINING_SEGMENTS:
    idx = rng.choice(X_train.shape[0], size=MAX_TRAINING_SEGMENTS, replace=False)
    X_train = X_train[idx]
    Y_train = Y_train[idx]
    print(f"Subsampled to {X_train.shape[0]} segments for faster training.")

print(f"\nFinal training set: {X_train.shape[0]} segments × {X_train.shape[1]} features, {Y_train.shape[1]} classes.")
print()
print("Class distribution — fraction of segments where class is active:")
for i, cls in enumerate(CLASS_NAMES):
    frac = Y_train[:, i].mean()
    bar  = "#" * int(frac * 40)
    print(f"  {cls:<35} {frac:.3f}  {bar}")

<a id='5'></a>
## 5. Training the Classifier

### The `MultiOutputClassifier` wrapper

In our setting, each audio segment can belong to **multiple sound event classes simultaneously** — this is a *multi-label* classification problem. For example, a segment might contain both `keyboard_typing` and `phone_ringing` at the same time.

A common and straightforward approach is the **binary relevance** strategy: train one **independent binary classifier per class**. Each classifier asks a simple yes/no question: "Is class X active in this segment?" Running all 15 classifiers on a segment yields a 15-dimensional binary prediction vector.

Scikit-learn's `MultiOutputClassifier` implements this pattern as a convenient wrapper. It takes any binary classifier as a base estimator and fits **one independent copy per output column** (i.e., per class). During prediction, the results of all sub-classifiers are stacked into a single prediction matrix of shape `(N_samples, N_classes)`.

For the baseline we use `DecisionTreeClassifier` as the base estimator — the same type of classifier you explored in the classification tutorial. We therefore train **15 independent decision trees**, one per sound event class.

> **Note on `max_features`**: Our feature vectors have 960 dimensions. A standard decision tree evaluates all 960 features at every split node, which is very slow. Setting `max_features='sqrt'` tells sklearn to consider only a random subset of ≈31 features at each split. This reduces training time by about 30× with minimal loss in accuracy — it is the same trick used inside Random Forests. You can set `max_features=None` to consider all features (slower but potentially better), or experiment with other values like `max_features=100`.

> **Note on polyphony**: Because the 15 classifiers are independent, multiple classes can (and should) be predicted as active at the same time. This is exactly what we want for polyphonic SED.

print("Training one DecisionTreeClassifier per class via MultiOutputClassifier...")
# max_features='sqrt' considers ~31 features per split instead of all 960.
# This speeds up training by ~30x while maintaining reasonable accuracy.
# max_depth=20 limits tree depth to avoid extremely long training times.
base_clf = DecisionTreeClassifier(max_depth=20, max_features="sqrt", random_state=42)
clf = MultiOutputClassifier(base_clf, n_jobs=-1)  # n_jobs=-1 uses all CPU cores in parallel
clf.fit(X_train, Y_train)
print(f"Training complete. Fitted {len(clf.estimators_)} sub-classifiers (one per class).")

# ── Hyperparameters ────────────────────────────────────────────────────────────
INPUT_DIM   = X_train.shape[1]  # number of audio features
OUTPUT_DIM  = y_train.shape[1]  # 15 labels
HIDDEN_DIMS = [512, 256, 128, 64]    # MLP layer sizes — tune these
DROPOUT     = 0.3               # dropout rate
BATCH_SIZE  = 64
EPOCHS      = 50
LR          = 1e-3
MODEL_PATH  = "best_nn_model.pt"


# ── Hyperparameters ────────────────────────────────────────────────────────────
INPUT_DIM   = X_train.shape[1]  # number of audio features
OUTPUT_DIM  = y_train.shape[1]  # 15 labels
HIDDEN_DIMS = [256, 128, 64]    # MLP layer sizes — tune these
DROPOUT     = 0.0              # dropout rate
BATCH_SIZE  = 64
EPOCHS      = 50
LR          = 1e-3
MODEL_PATH  = "best_nn_model.pt"

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

# ══════════════════════════════════════════════════════════════════════════════
# 0. DATA  —  fill in your pre-split sets here
# ══════════════════════════════════════════════════════════════════════════════
X_train, y_train = load_all_segments(train_files_to_use)
X_val,   y_val   = load_all_segments(our_val_files)
X_test,  y_test  = load_all_segments(our_test_files)

# ── Hyperparameters ────────────────────────────────────────────────────────────
INPUT_DIM   = X_train.shape[1]  # number of audio features
OUTPUT_DIM  = y_train.shape[1]  # 15 labels
HIDDEN_DIMS = [256, 128, 64]    # MLP layer sizes — tune these
DROPOUT     = 0.0              # dropout rate
BATCH_SIZE  = 64
EPOCHS      = 50
LR          = 1e-3
MODEL_PATH  = "best_nn_model.pt"

# ══════════════════════════════════════════════════════════════════════════════
# 1. DEVICE
# ══════════════════════════════════════════════════════════════════════════════
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ══════════════════════════════════════════════════════════════════════════════
# 2. DATALOADERS
# ══════════════════════════════════════════════════════════════════════════════
def make_loader(X, y, batch_size, shuffle=True):
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.float32)   # float for BCEWithLogitsLoss
    return DataLoader(TensorDataset(X_t, y_t),
                      batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train, y_train, BATCH_SIZE, shuffle=True)
val_loader   = make_loader(X_val,   y_val,   BATCH_SIZE, shuffle=False)

# ══════════════════════════════════════════════════════════════════════════════
# 3. MODEL
# ══════════════════════════════════════════════════════════════════════════════
class AudioMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dim, dropout):
        super().__init__()

        layers = []
        prev_dim = input_dim
        for h_dim in hidden_dims:
            layers += [
                nn.Linear(prev_dim, h_dim),
                nn.BatchNorm1d(h_dim),   # stabilises training
                nn.ReLU(),
                nn.Dropout(dropout),
            ]
            prev_dim = h_dim

        layers.append(nn.Linear(prev_dim, output_dim))  # no activation → raw logits
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

model = AudioMLP(INPUT_DIM, HIDDEN_DIMS, OUTPUT_DIM, DROPOUT).to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

# ══════════════════════════════════════════════════════════════════════════════
# 4. LOSS & OPTIMISER
# ══════════════════════════════════════════════════════════════════════════════
# BCEWithLogitsLoss = sigmoid + binary cross entropy, per label independently
# pos_weight handles class imbalance: upweights the positive class
pos_counts  = torch.tensor(y_train.sum(axis=0),            dtype=torch.float32)
neg_counts  = torch.tensor(len(y_train) - y_train.sum(axis=0), dtype=torch.float32)
pos_weight  = (neg_counts / pos_counts.clamp(min=1)).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", patience=5, factor=0.5
)

# ══════════════════════════════════════════════════════════════════════════════
# 5. TRAINING LOOP
# ══════════════════════════════════════════════════════════════════════════════
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    with torch.set_grad_enabled(train):
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)
            loss   = criterion(logits, y_batch)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(X_batch)
    return total_loss / len(loader.dataset)

train_losses, val_losses = [], []
best_val_loss = float("inf")
cool_epoch = 0
print("\n── Training ──────────────────────────────────────────────────────────")
for epoch in range(1, EPOCHS + 1):
    tr_loss  = run_epoch(train_loader, train=True)
    val_loss = run_epoch(val_loader,   train=False)
    scheduler.step(val_loss)

    train_losses.append(tr_loss)
    val_losses.append(val_loss)

    # Save best model checkpoint
    if val_loss < best_val_loss:
        cool_epoch = epoch
        best_val_loss = val_loss
        torch.save(model.state_dict(), MODEL_PATH)
        tag = "  ✓ saved"
    else:
        tag = ""

    if epoch - cool_epoch >10:
        break
    print(f"Epoch {epoch:>3}/{EPOCHS}  "
              f"train_loss={tr_loss:.4f}  val_loss={val_loss:.4f}{tag}")

print(f"\nBest val loss: {best_val_loss:.4f}  →  model saved to '{MODEL_PATH}'")

# ══════════════════════════════════════════════════════════════════════════════
# 6. LEARNING CURVES
# ══════════════════════════════════════════════════════════════════════════════
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label="Train loss", color="steelblue")
plt.plot(val_losses,   label="Val loss",   color="darkorange")
plt.xlabel("Epoch")
plt.ylabel("BCE Loss")
plt.title("Neural Network — Learning Curves")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("nn_learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))

In [ ]:
MODEL_PATH  = "best_nn_model.pt"
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))

print("Training one DecisionTreeClassifier per class via MultiOutputClassifier...")
# max_features='sqrt' considers ~31 features per split instead of all 960.
# This speeds up training by ~30x while maintaining reasonable accuracy.
# max_depth=20 limits tree depth to avoid extremely long training times.
base_clf = DecisionTreeClassifier(max_depth=20, max_features="sqrt", random_state=42)
clf = MultiOutputClassifier(base_clf, n_jobs=-1)  # n_jobs=-1 uses all CPU cores in parallel
clf.fit(X_train, Y_train)
print(f"Training complete. Fitted {len(clf.estimators_)} sub-classifiers (one per class).")

<a id='6'></a>
## 6. Sound Event Detection Inference

### Using segment classifiers for full-recording SED

In the previous project phase, classifiers operated on isolated 1-second segments and simply predicted which classes were active in each segment. For the challenge, we need to detect events **within complete recordings of arbitrary length**, including their onset and offset timestamps.

Although state-of-the-art SED systems rely on sophisticated neural architectures, a simple yet effective approach is to reuse the segment classifiers from the previous phase. The pipeline works as follows:

**Step 1, 2:** **Segmentation**: Each audio recording is divided into overlapping 1-second windows using a hop size of 0.5 s. This produces segments starting at t = 0.0, 0.5, 1.0, 1.5, 2.0, ... seconds (see **step 2** in the visualization below). Because the features are precomputed, no audio processing is needed.

**Step 3:** **Classification**: All segments are passed through the 15 binary classifiers (via `MultiOutputClassifier`). Each segment receives a 15-dimensional binary prediction vector.

**Step 4**: **Temporal aggregation**: The challenge evaluation metric operates at **1-second resolution**, so exactly one prediction per second is needed. As the simplest possible strategy, we **keep only the predictions for segments starting at whole-second timestamps** (t = 0.0, 1.0, 2.0, ...) and discard the half-second predictions (t = 0.5, 1.5, ...).

   > The half-second predictions are discarded here for simplicity. A natural extension would be to use them for smoothing (e.g., average or max-pool adjacent pairs) or for sub-second temporal refinement.

**Step 5:** **Post-processing**: The per-second binary predictions are converted to onset/offset intervals (the required submission format). Consecutive active seconds for the same class are merged into a single interval.

**Here is a visualization of this pipeline:**

<img src="imgs/classifiers_for_SED_upscaled.png">

def run_sed_inference(
    filepath: str,
    classifier: MultiOutputClassifier,
) -> Tuple[np.ndarray, np.ndarray, str]:
    """Run SED inference on a single recording.

    Loads all overlapping segments, applies the classifier to each of them,
    and returns predictions only for segments starting at whole-second timestamps.

    Parameters
    ----------
    filepath : str
        Path to the .npz feature file for one recording.
    classifier : MultiOutputClassifier
        A fitted multi-label classifier.

    Returns
    -------
    predictions : np.ndarray of shape (N_whole_second_segments, N_classes)
        Binary predictions (0 or 1) for each class at each whole-second segment.
    start_times : np.ndarray of shape (N_whole_second_segments,)
        Start times (in seconds) of the retained segments.
    filename : str
        Audio filename for this recording (e.g. '000001.wav').
    """
    data = dict(np.load(filepath, allow_pickle=True))
    start_times_all = data["start_time"]   # [0.0, 0.5, 1.0, 1.5, ...]

    # Build feature matrix for ALL segments (both whole-second and half-second)
    X_all = build_feature_matrix(data)     # shape: (N_all_segments, D_features)

    # Apply the classifier to every segment
    pred_all = classifier.predict(X_all)   # shape: (N_all_segments, N_classes)

    # Identify segments that start at a whole-second timestamp (0.0, 1.0, 2.0, ...)
    whole_second_mask = np.isclose(start_times_all % 1.0, 0.0)
    # The half-second segments (0.5, 1.5, ...) are dropped here.
    # Future improvement: use them to smooth predictions or refine temporal boundaries.

    predictions = pred_all[whole_second_mask]
    start_times = start_times_all[whole_second_mask]

    filename = os.path.basename(filepath).replace(".npz", ".wav")
    return predictions, start_times, filename

In [ ]:
def predict(model, X, batch_size=256,THRESHOLD = 0.5):
    """Run model on numpy array X, return binary predictions and probabilities."""
    loader  = DataLoader(
        TensorDataset(torch.tensor(X, dtype=torch.float32)),
        batch_size=batch_size, shuffle=False
    )
    all_probs = []
    with torch.no_grad():
        for (X_batch,) in loader:
            logits = model(X_batch.to(device))
            probs  = torch.sigmoid(logits).cpu().numpy()
            all_probs.append(probs)
    probs = np.vstack(all_probs)
    preds = (probs >= THRESHOLD).astype(int)
    return preds, probs

In [ ]:
def run_sed_inference(
    filepath: str,
    classifier: MultiOutputClassifier,
) -> Tuple[np.ndarray, np.ndarray, str]:
    """Run SED inference on a single recording.

    Loads all overlapping segments, applies the classifier to each of them,
    and returns predictions only for segments starting at whole-second timestamps.

    Parameters
    ----------
    filepath : str
        Path to the .npz feature file for one recording.
    classifier : MultiOutputClassifier
        A fitted multi-label classifier.

    Returns
    -------
    predictions : np.ndarray of shape (N_whole_second_segments, N_classes)
        Binary predictions (0 or 1) for each class at each whole-second segment.
    start_times : np.ndarray of shape (N_whole_second_segments,)
        Start times (in seconds) of the retained segments.
    filename : str
        Audio filename for this recording (e.g. '000001.wav').
    """
    data = dict(np.load(filepath, allow_pickle=True))
    start_times_all = data["start_time"]   # [0.0, 0.5, 1.0, 1.5, ...]

    # Build feature matrix for ALL segments (both whole-second and half-second)
    X_all = build_feature_matrix(data)     # shape: (N_all_segments, D_features)

    # Apply the classifier to every segment
    # pred_all = classifier.predict(X_all)   # shape: (N_all_segments, N_classes)
    pred_all = predict(model,X_all,batch_size=64)[0]

    # Identify segments that start at a whole-second timestamp (0.0, 1.0, 2.0, ...)
    whole_second_mask = np.isclose(start_times_all % 1.0, 0.0)
    # The half-second segments (0.5, 1.5, ...) are dropped here.
    # Future improvement: use them to smooth predictions or refine temporal boundaries.

    predictions = pred_all[whole_second_mask]
    start_times = start_times_all[whole_second_mask]

    filename = os.path.basename(filepath).replace(".npz", ".wav")
    return predictions, start_times, filename

In [ ]:
def predictions_to_intervals(
    predictions: np.ndarray,
    start_times: np.ndarray,
    filename: str,
) -> List[Dict]:
    """Convert per-second binary predictions to onset/offset interval annotations.

    Consecutive active 1-second segments are merged into a single interval.
    For example, predictions active at t = 2 s and t = 3 s are merged into
    the interval [2.0, 4.0] (both 1-second windows combined).

    Parameters
    ----------
    predictions : np.ndarray of shape (N_segments, N_classes)
        Binary predictions for each class at each whole-second segment.
    start_times : np.ndarray of shape (N_segments,)
        Start time (in seconds) of each segment.
    filename : str
        Audio filename (e.g. '000001.wav').

    Returns
    -------
    rows : list of dict with keys: 'filename', 'annotation', 'onset', 'offset'
    """
    rows = []

    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        cls_preds = predictions[:, cls_idx]

        in_event = False
        onset    = None

        for t, pred in zip(start_times, cls_preds):
            if pred == 1 and not in_event:
                onset    = float(t)
                in_event = True
            elif pred == 0 and in_event:
                # The event ends at the start of the first inactive segment,
                # which equals the start time of the previous active segment + 1 s.
                offset = float(t)
                rows.append({"filename": filename, "annotation": cls_name,
                             "onset": onset, "offset": offset})
                in_event = False

        # Handle an event that is still active at the very end of the recording
        if in_event:
            offset = float(start_times[-1]) + SEGMENT_LENGTH
            rows.append({"filename": filename, "annotation": cls_name,
                         "onset": onset, "offset": offset})

    return rows


def generate_predictions(
    file_list: List[str],
    classifier: MultiOutputClassifier,
) -> pd.DataFrame:
    """Run SED inference on a list of recordings and return predictions as a DataFrame.

    Parameters
    ----------
    file_list : list of str
        Paths to .npz feature files.
    classifier : MultiOutputClassifier
        A fitted multi-label classifier.

    Returns
    -------
    pred_df : pd.DataFrame
        Columns: filename, annotation, onset, offset.
        One row per detected event interval.
        Recordings with no detected events are not represented.
    """
    all_rows = []
    for filepath in file_list:
        preds, times, fname = run_sed_inference(filepath, classifier)
        all_rows.extend(predictions_to_intervals(preds, times, fname))

    if not all_rows:
        return pd.DataFrame(columns=["filename", "annotation", "onset", "offset"])
    return pd.DataFrame(all_rows)

In [ ]:
# Quick demo: run inference on one recording and inspect the output.
# We use a file from the non-hidden test set so we have ground-truth labels for comparison.
demo_file = our_test_files[7]
example_preds, example_times, example_fname = run_sed_inference(demo_file, clf)

print(f"File: {example_fname}")
print(f"Whole-second segments: {len(example_times)}  (t = {example_times[0]:.0f} s ... {example_times[-1]:.0f} s)")
print()
print("Predicted active classes per second:")
for i in range(len(example_times)):
    active = [CLASS_NAMES[c] for c in range(len(CLASS_NAMES)) if example_preds[i, c] == 1]
    print(f"  t = {example_times[i]:.0f} s : {active if active else '(none)'}")

print()
print("Converted to intervals:")
example_rows = predictions_to_intervals(example_preds, example_times, example_fname)
for row in example_rows[:10]:
    print(f"  {row['annotation']:<35} onset={row['onset']:.1f}  offset={row['offset']:.1f}")

# ---------------------------------------------------------------------------
# Visualization: mel spectrogram + ground truth vs. predictions
# ---------------------------------------------------------------------------
demo_data        = dict(np.load(demo_file, allow_pickle=True))
demo_start_times = demo_data["start_time"]                     # [0.0, 0.5, 1.0, ...]

# Whole-second mask — same filter used in run_sed_inference
whole_second_mask = np.isclose(demo_start_times % 1.0, 0.0)

# Ground-truth labels at whole-second resolution (majority-vote across annotators)
gt_all   = get_segment_labels(demo_data)    # (N_all_segments, C)
gt_whole = gt_all[whole_second_mask]        # (N_ws, C)

# Log-mel spectrogram: use ALL segments (hop=0.5 s) for twice the time resolution
mel_log = np.log1p(demo_data["melspect_mean"].T)  # (n_mels, N_all)

C = len(CLASS_NAMES)

# pcolormesh edge arrays
t_edges_all = np.append(demo_start_times, demo_start_times[-1] + HOP_SIZE)
t_edges_ws  = np.append(example_times,    float(example_times[-1]) + SEGMENT_LENGTH)
mel_edges   = np.arange(mel_log.shape[0] + 1) - 0.5
cls_edges   = np.arange(C + 1) - 0.5

fig, axes = plt.subplots(3, 1, figsize=(16, 11),
                          gridspec_kw={"height_ratios": [2, 1.5, 1.5]},
                          sharex=True)
fig.suptitle(f"Recording: {example_fname}", fontsize=13, fontweight="bold")

# --- 1. Log-mel spectrogram ---
ax0 = axes[0]
pcm = ax0.pcolormesh(t_edges_all, mel_edges, mel_log, cmap="magma", shading="flat")
fig.colorbar(pcm, ax=ax0, label="log(1 + mean energy)", pad=0.01)
ax0.set_ylabel("Mel bin")
ax0.set_title("Log mel spectrogram (segment means, hop = 0.5 s)")

# --- 2. Ground truth ---
ax1 = axes[1]
ax1.pcolormesh(t_edges_ws, cls_edges, gt_whole.T,
               cmap="Greens", vmin=0, vmax=1, shading="flat")
ax1.set_yticks(range(C))
ax1.set_yticklabels(CLASS_NAMES, fontsize=8)
ax1.invert_yaxis()
ax1.set_title("Ground Truth")
ax1.grid(axis="x", linestyle="--", linewidth=0.6, alpha=0.4, color="gray")
ax1.set_ylabel("Class")

# --- 3. Predictions ---
ax2 = axes[2]
ax2.pcolormesh(t_edges_ws, cls_edges, example_preds.T,
               cmap="Oranges", vmin=0, vmax=1, shading="flat")
ax2.set_yticks(range(C))
ax2.set_yticklabels(CLASS_NAMES, fontsize=8)
ax2.invert_yaxis()
ax2.set_title("Predictions")
ax2.grid(axis="x", linestyle="--", linewidth=0.6, alpha=0.4, color="gray")
ax2.set_xlabel("Time (s)")
ax2.set_ylabel("Class")

plt.tight_layout()
plt.show()

In [ ]:
# --- Ground truth + predictions (same format) ---
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap



fig, axes = plt.subplots(2, 1, figsize=(16, 11),
                          gridspec_kw={"height_ratios": [2, 1.5]},
                          sharex=True)
fig.suptitle(f"Recording: {example_fname}", fontsize=13, fontweight="bold")

# --- 1. Log-mel spectrogram ---
ax0 = axes[0]
pcm = ax0.pcolormesh(t_edges_all, mel_edges, mel_log, cmap="magma", shading="flat")
fig.colorbar(pcm, ax=ax0, label="log(1 + mean energy)", pad=0.01)
ax0.set_ylabel("Mel bin")
ax0.set_title("Log mel spectrogram (segment means, hop = 0.5 s)")




true = gt_whole.T
pred = example_preds.T
cmap = LinearSegmentedColormap.from_list(
    "red_white_green",
    ["red", "white", "green"]
)
# --- Build visualization matrix ---
viz = np.zeros_like(true, dtype=float)

tp = (true == 1) & (pred == 1)   # correct
fp = (true == 0) & (pred == 1)   # false alarm
fn = (true == 1) & (pred == 0)   # missed

viz[tp] = 1
viz[fp] = -1
viz[fn] = -1

# --- Plot ---
ax = axes[1]

im = ax.pcolormesh(
    t_edges_ws,
    np.arange(len(CLASS_NAMES) + 1) - 0.5,
    viz,
    cmap=cmap,
    vmin=-1,
    vmax=1,
    shading="flat"
)

ax.set_yticks(range(len(CLASS_NAMES)))
ax.set_yticklabels(CLASS_NAMES, fontsize=8)
ax.invert_yaxis()

ax.set_title(f"Prediction vs Ground Truth — {example_fname}")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Class")



green_patch = mpatches.Patch(color=cmap(0.9), label="Correct (TP)")
red_patch   = mpatches.Patch(color=cmap(0.1), label="Error (FP/FN)")

ax.legend(handles=[green_patch, red_patch], loc="upper right")

# plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(f"{example_fname}.png")
plt.show()

<a id='7'></a>
## 7. Evaluation

### Evaluation metric

The challenge uses **segment-level macro F1 score**:

1. Both ground-truth annotations and predictions are converted to a binary segment-level representation at 1-second resolution. Each `(filename, second)` pair is independently labeled as active or inactive per class.
2. For each of the 15 classes, standard binary precision, recall, and F1 are computed.
3. The **macro F1** is the unweighted average F1 across all 15 classes.

We reuse the helper functions from the provided `evaluate.py` script:
- `aggregate_ground_truth_annotations`: applies majority vote across annotators.
- `build_segment_frame_from_intervals`: expands onset/offset intervals into per-second binary labels.
- `calculate_f1_score`: computes per-class and macro F1.

### Evaluation procedure

We evaluate on two local splits:
- **Local validation set**: use to iterate on your model.
- **Non-hidden test set**: a one-shot evaluation — do not use this result to tune your model.

In [ ]:
# Load the ground-truth annotations for the validation split.
# (These cover all ~999 validation recordings; we will filter to the relevant subset.)
ann_df = pd.read_csv(os.path.join(PATH_VAL, "annotations.csv"))
print(f"Loaded {len(ann_df)} annotation rows for {ann_df['filename'].nunique()} recordings.")


def evaluate_split(
    pred_df: pd.DataFrame,
    file_list: List[str],
    ann_df: pd.DataFrame,
) -> Tuple[float, pd.DataFrame]:
    """Evaluate SED predictions against ground-truth annotations for a given split.

    Parameters
    ----------
    pred_df : pd.DataFrame
        Prediction DataFrame with columns: filename, annotation, onset, offset.
    file_list : list of str
        Paths to .npz files for the split to evaluate.
    ann_df : pd.DataFrame
        Full annotation DataFrame (loaded from annotations.csv); will be filtered.

    Returns
    -------
    macro_f1 : float
    results : pd.DataFrame
        Per-class precision, recall, and F1 score.
    """
    split_filenames = {os.path.basename(f).replace(".npz", ".wav") for f in file_list}

    # Filter annotations to only include files in this split
    ann_split = ann_df[ann_df["filename"].isin(split_filenames)].copy()

    # Aggregate per-annotator annotations via majority vote
    gt = aggregate_ground_truth_annotations(ann_split)

    # Build segment-level indicator matrices (indexed by filename + second)
    gt_segments   = build_segment_frame_from_intervals(gt,      name="ground_truth")
    pred_segments = build_segment_frame_from_intervals(pred_df, name="predictions")

    # Restrict predictions to files belonging to this split
    if len(pred_segments) > 0:
        pred_filenames = pred_segments.index.get_level_values("filename")
        pred_segments  = pred_segments[pred_filenames.isin(split_filenames)]

    return calculate_f1_score(gt_segments, pred_segments)

In [ ]:
print("Generating predictions for the local validation set...")
pred_val_df = generate_predictions(our_val_files, clf)
print(f"  {len(pred_val_df)} event intervals predicted across "
      f"{pred_val_df['filename'].nunique() if len(pred_val_df) > 0 else 0} files.")

macro_f1_val, results_val = evaluate_split(pred_val_df, our_val_files, ann_df)

print(f"\n=== Local Validation Set — Macro F1: {macro_f1_val:.4f} ===")
print()
print(results_val.to_string(index=False))

In [ ]:
print("Generating predictions for the non-hidden test set...")
pred_nht_df = generate_predictions(our_test_files, clf)
print(f"  {len(pred_nht_df)} event intervals predicted across "
      f"{pred_nht_df['filename'].nunique() if len(pred_nht_df) > 0 else 0} files.")

macro_f1_nht, results_nht = evaluate_split(pred_nht_df, our_test_files, ann_df)

print(f"\n=== Non-Hidden Test Set — Macro F1: {macro_f1_nht:.4f} ===")
print()
print(results_nht.to_string(index=False))

In [ ]:
def smooth_predictions(pred_df, alpha):
    smoothed = pred_df.copy()
    merged_rows = []

    for (filename, annotation), group in smoothed.groupby(["filename", "annotation"]):
        group = group.sort_values("onset")

        current_onset = None
        current_offset = None

        for _, row in group.iterrows():
            if current_onset is None:
                current_onset = row["onset"]
                current_offset = row["offset"]
            elif row["onset"] - current_offset <= alpha:
                current_offset = max(current_offset, row["offset"])
            else:
                merged_rows.append({
                    "filename": filename,
                    "annotation": annotation,
                    "onset": current_onset,
                    "offset": current_offset
                })
                current_onset = row["onset"]
                current_offset = row["offset"]

        if current_onset is not None:
            merged_rows.append({
                "filename": filename,
                "annotation": annotation,
                "onset": current_onset,
                "offset": current_offset
            })

    return pd.DataFrame(merged_rows)

In [ ]:
smoothing_results = []

smoothing_results.append({
    "method": "No post-processing",
    "macro_f1": macro_f1_nht
})

for alpha in [0.5, 1.0, 2.0]:
    pred_smooth = smooth_predictions(pred_nht_df, alpha)

    macro_f1_smooth, results_smooth = evaluate_split(
        pred_smooth,
        our_test_files,
        ann_df
    )

    smoothing_results.append({
        "method": f"Temporal smoothing, alpha = {alpha}",
        "macro_f1": macro_f1_smooth
    })

    print(f"Temporal smoothing alpha={alpha}: Macro F1 = {macro_f1_smooth:.4f}")

smoothing_results_df = pd.DataFrame(smoothing_results)
smoothing_results_df

In [ ]:
# Visualize per-class F1 scores for both local evaluation splits
fig, ax = plt.subplots(figsize=(12, 5))
x     = np.arange(len(CLASS_NAMES))
width = 0.38

ax.bar(x - width / 2, results_val["f1"], width, label=f"Validation (macro={macro_f1_val:.3f})",  color="steelblue")
ax.bar(x + width / 2, results_nht["f1"], width, label=f"Non-hidden test (macro={macro_f1_nht:.3f})", color="coral")

ax.axhline(macro_f1_val, color="steelblue", linestyle="--", linewidth=0.9, alpha=0.7)
ax.axhline(macro_f1_nht, color="coral",     linestyle="--", linewidth=0.9, alpha=0.7)

ax.set_xticks(x)
ax.set_xticklabels(results_val["annotation"], rotation=45, ha="right")
ax.set_ylabel("F1 Score")
ax.set_ylim(0, 1.05)
ax.set_title("Per-Class F1 Score — Baseline Decision Tree")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Validation macro F1:      {macro_f1_val:.4f}")
print(f"Non-hidden test macro F1: {macro_f1_nht:.4f}")

# trees
Validation macro F1:      0.3072
Non-hidden test macro F1: 0.3170
# detailed classes macro-f1 score
0 	 0.24262295081967214 	 0.28823529411764703
1 	 0.2910284463894967 	 0.36200716845878134
2 	 0.30111524163568776 	 0.2872512103281334
3 	 0.1588271227855834 	 0.17733411626541398
4 	 0.3254953764861295 	 0.33275734386571215
5 	 0.37724322260404736 	 0.35895305359368507
6 	 0.3433420365535248 	 0.32114882506527415
7 	 0.2169811320754717 	 0.23366336633663368
8 	 0.41731066460587324 	 0.38961038961038963
9 	 0.4450900639163277 	 0.48484848484848486
10 	 0.616557161629435 	     0.5775695783962524
11 	 0.28475336322869954 	 0.2903533906399236
12 	 0.4332129963898917 	 0.46974236069502695
13 	 0.10053475935828877 	 0.12026726057906459
14 	 0.05333333333333334 	 0.06109324758842444

Trees


Generating predictions for the non-hidden test set...
  8022 event intervals predicted across 500 files.

=== Non-Hidden Test Set — Macro F1: 0.3170 ===

                annotation  precision   recall       f1  map
              bell_ringing   0.426087 0.217778 0.288235 None
            coffee_machine   0.503741 0.282517 0.362007 None
            cutlery_dishes   0.294053 0.280757 0.287251 None
           door_open_close   0.200265 0.159115 0.177334 None
                 footsteps   0.384265 0.293426 0.332757 None
           keyboard_typing   0.372093 0.346709 0.358953 None
                  keychain   0.364985 0.286713 0.321149 None
              light_switch   0.239837 0.227799 0.233663 None
                 microwave   0.407911 0.372881 0.389610 None
             phone_ringing   0.533969 0.444004 0.484848 None
             running_water   0.655000 0.516511 0.577570 None
           toilet_flushing   0.288425 0.292308 0.290353 None
            vacuum_cleaner   0.485149 0.455285 0.469742 None
wardrobe_drawer_open_close   0.133663 0.109312 0.120267 None
         window_open_close   0.056716 0.066202 0.061093 None
        


NN 3 layers, 0.0 drop
Generating predictions for the non-hidden test set...
  14002 event intervals predicted across 500 files.

=== Non-Hidden Test Set — Macro F1: 0.4175 ===

                annotation  precision   recall       f1  map
              bell_ringing   0.204412 0.617778 0.307182 None
            coffee_machine   0.224619 0.742657 0.344917 None
            cutlery_dishes   0.387393 0.710831 0.501484 None
           door_open_close   0.277369 0.715490 0.399764 None
                 footsteps   0.464328 0.685677 0.553700 None
           keyboard_typing   0.423795 0.783307 0.550014 None
                  keychain   0.321608 0.745921 0.449438 None
              light_switch   0.148953 0.741313 0.248062 None
                 microwave   0.384566 0.681356 0.491643 None
             phone_ringing   0.505401 0.649158 0.568330 None
             running_water   0.594662 0.823558 0.690639 None
           toilet_flushing   0.148785 0.848077 0.253157 None
            vacuum_cleaner   0.421773 0.823461 0.557828 None
wardrobe_drawer_open_close   0.119438 0.722672 0.204996 None
         window_open_close   0.078107 0.724739 0.141017 None

NN 4 layers, 0.3 drop
Generating predictions for the non-hidden test set...
  14851 event intervals predicted across 500 files.

=== Non-Hidden Test Set — Macro F1: 0.3854 ===

                annotation  precision   recall       f1  map
              bell_ringing   0.119898 0.835556 0.209704 None
            coffee_machine   0.175141 0.737063 0.283029 None
            cutlery_dishes   0.269394 0.803365 0.403486 None
           door_open_close   0.230432 0.775553 0.355298 None
                 footsteps   0.397612 0.768394 0.524050 None
           keyboard_typing   0.390644 0.764045 0.516970 None
                  keychain   0.320332 0.765734 0.451702 None
              light_switch   0.135135 0.772201 0.230017 None
                 microwave   0.246508 0.797740 0.376634 None
             phone_ringing   0.349593 0.724480 0.471613 None
             running_water   0.695949 0.753573 0.723616 None
           toilet_flushing   0.244558 0.734615 0.366955 None
            vacuum_cleaner   0.413754 0.838560 0.554106 None
wardrobe_drawer_open_close   0.109741 0.720648 0.190476 None
         window_open_close   0.067100 0.756098 0.123260 None

Neural (first attempt)
Generating predictions for the non-hidden test set...
  14898 event intervals predicted across 500 files.

=== Non-Hidden Test Set — Macro F1: 0.3927 ===

                annotation  precision   recall       f1  map
              bell_ringing   0.130752 0.795556 0.224592 None
            coffee_machine   0.206719 0.731469 0.322342 None
            cutlery_dishes   0.272466 0.797056 0.406108 None
           door_open_close   0.228723 0.770285 0.352714 None
                 footsteps   0.396451 0.797562 0.529633 None
           keyboard_typing   0.431716 0.771268 0.553571 None
                  keychain   0.324418 0.763403 0.455335 None
              light_switch   0.263909 0.714286 0.385417 None
                 microwave   0.243971 0.788701 0.372664 None
             phone_ringing   0.337772 0.754212 0.466585 None
             running_water   0.564060 0.846230 0.676917 None
           toilet_flushing   0.168396 0.811538 0.278916 None
            vacuum_cleaner   0.413872 0.845528 0.555725 None
wardrobe_drawer_open_close   0.103153 0.781377 0.182247 None
         window_open_close   0.069462 0.773519 0.127476 None

In [ ]:
for i in range(len(results_val["f1"])):
    print(i,"\t",results_val["f1"][i],"\t", results_nht["f1"][i])

<a id='8'></a>
## 8. Generating Predictions for the Hidden Test Set

Once you are satisfied with your system's local performance, generate predictions for the hidden test set and save them as a CSV file for submission.

The hidden test set has no annotations — local evaluation is not possible. Final rankings are determined by the teaching team using the macro F1 score on this set.

### Submission format

The output CSV must have exactly these columns:

| Column | Description |
|--------|-------------|
| `filename` | Audio filename, e.g. `000112.wav` |
| `annotation` | Predicted sound event class (must be one of the 15 class names) |
| `onset` | Start time of the detected event in seconds |
| `offset` | End time of the detected event in seconds |

Multiple rows can share the same `filename` (one row per detected event interval). Files with no detected events are simply omitted from the CSV.

Example:
```
filename,annotation,onset,offset
000001.wav,keyboard_typing,0.0,21.0
000001.wav,phone_ringing,9.0,18.0
000112.wav,footsteps,3.0,7.0
```

In [ ]:
print(f"Generating predictions for {len(test_files)} hidden test recordings...")
pred_test_df = generate_predictions(test_files, clf)
print(f"Done. {len(pred_test_df)} event intervals across "
      f"{pred_test_df['filename'].nunique() if len(pred_test_df) > 0 else 0} files.")

pred_test_df.head(10)

In [ ]:
OUTPUT_CSV = "predictions_hidden_test.csv"
pred_test_df.to_csv(OUTPUT_CSV, index=False)
print(f"Predictions saved to: {OUTPUT_CSV}")
print(f"Total rows: {len(pred_test_df)}")
print(f"Classes predicted: {sorted(pred_test_df['annotation'].unique()) if len(pred_test_df) > 0 else []}")